        # 📈 L11　matplotlib 繪製圖表
        **Python 冒險之旅 2026**　｜　Day 5（09/04 五）🏜️ 檔案之島　｜　關卡　｜　🏅 100 XP

        📖 對應教科書：第 10 章 10.1–10.4


        ### 🎯 這一關你會學到
        - 繪製線條圖、柱狀圖、圓餅圖
- 設定標題、座標、圖例與中文字型
- 在 Colab 中正確顯示與儲存圖表

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "L11"
_SALT = "python-quest-2026-datama"
_TASKS = ["11-1", "11-2", "11-3", "11-4", "11-5"]
_XP_EACH = 20
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_11_1(run):
    out, ns = run()
    import matplotlib.pyplot as plt
    ax = plt.gcf().axes[0] if plt.gcf().axes else None
    if ax is None or len(ax.lines) != 1: return (False, "應該剛好畫一條線。")
    ln = ax.lines[0]
    if ln.get_linestyle() != '--': return (False, "線條樣式要是虛線 '--'。")
    if ln.get_marker() != 'o': return (False, "marker 要是 'o'。")
    if ax.get_title() != '我的第一張圖': return (False, "標題要是 我的第一張圖。")
    ok = ax.get_legend() is not None
    plt.close('all')
    return (ok, "要呼叫 plt.legend() 顯示圖例。")
任務定義("11-1", _check_11_1, 提示="plt.plot(x, y, color='red', ls='--', marker='o', label='y = 2x')。")

def _check_11_2(run):
    out, ns = run()
    import matplotlib.pyplot as plt
    ax = plt.gcf().axes[0] if plt.gcf().axes else None
    if ax is None or len(ax.lines) != 3: return (False, "應該畫 3 條線。")
    labels = [l.get_label() for l in ax.lines]
    if not {'商管', '設計', '資電'} <= set(labels): return (False, "三條線的 label 要是 商管、設計、資電。")
    if ax.get_title() != '歷年錄取分數': return (False, "標題要是 歷年錄取分數。")
    ok = ax.get_xlabel() == '年度' and ax.get_ylabel() == '分數' and ax.get_legend() is not None
    plt.close('all')
    return (ok, "x 軸 年度、y 軸 分數，並顯示圖例。")
任務定義("11-2", _check_11_2, 提示="每一群用一個 plt.plot(years, 資料, label='...')。")

def _check_11_3(run):
    out, ns = run()
    import matplotlib.pyplot as plt
    ax = plt.gcf().axes[0] if plt.gcf().axes else None
    if ax is None or len(ax.patches) != 5: return (False, "應該有 5 根柱子（plt.bar）。")
    heights = sorted(p.get_height() for p in ax.patches)
    if heights != [55, 61, 78, 88, 92]: return (False, "柱子高度要對應分數。")
    if ax.get_title() != '期中考成績': return (False, "標題要是 期中考成績。")
    ok = ax.get_ylim()[0] == 0 and ax.get_ylim()[1] == 100
    plt.close('all')
    return (ok, "plt.ylim(0, 100)。")
任務定義("11-3", _check_11_3, 提示="plt.bar(names, scores)；plt.ylim(0, 100)。")

def _check_11_4(run):
    out, ns = run()
    import matplotlib.pyplot as plt
    ax = plt.gcf().axes[0] if plt.gcf().axes else None
    if ax is None or len(ax.patches) != 3: return (False, "應該有 3 塊（plt.pie）。")
    texts = [t.get_text() for t in ax.texts]
    if not any('%' in t for t in texts): return (False, "要用 autopct 顯示百分比。")
    if not {'商管', '設計', '資電'} <= set(texts): return (False, "labels 要是 商管、設計、資電。")
    ok = ax.get_title() == '招生員額比例'
    plt.close('all')
    return (ok, "標題要是 招生員額比例。")
任務定義("11-4", _check_11_4, 提示="plt.pie(ratio, labels=groups, autopct='%.1f%%', explode=(0, 0, 0.1))。")

def _check_11_5(run):
    import os
    if os.path.exists('square.png'): os.remove('square.png')
    out, ns = run()
    import matplotlib.pyplot as plt
    ax = plt.gcf().axes[0] if plt.gcf().axes else None
    ok_title = ax is not None and ax.get_title() == '平方數'
    plt.close('all')
    if not ok_title: return (False, "標題要是 平方數。")
    return (os.path.exists('square.png') and os.path.getsize('square.png') > 1000, "沒有產生 square.png，savefig 要在 show 之前。")
任務定義("11-5", _check_11_5, 提示="plt.savefig('square.png') 放在 plt.show() 前面。")


## 📈 11-1　matplotlib：資料視覺化的起點
`matplotlib.pyplot` 通常簡寫成 `plt`。在 Colab 裡 **不需要安裝**，圖會直接顯示在格子下方（課本 10.2.2 的問題不會發生）。
基本流程：**準備資料 → 畫圖（plot/bar/pie）→ 加標題座標圖例 → `plt.show()`**。

In [ ]:
#@title 🈶 中文字型設定（Colab 預設沒有中文字型，執行這一格安裝；約 20～40 秒）
import subprocess, glob, matplotlib
from matplotlib import font_manager
subprocess.run("apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1", shell=True)
for f in glob.glob('/usr/share/fonts/opentype/noto/NotoSansCJK*-Regular.ttc'):
    font_manager.fontManager.addfont(f)
matplotlib.rcParams['font.family'] = 'Noto Sans CJK JP'    # 這個字型檔同時包含繁體中文字形
matplotlib.rcParams['axes.unicode_minus'] = False          # 讓負號正常顯示
print("✅ 中文字型設定完成（若圖表中文仍是方框，請重新執行這一格後再畫一次）")

In [ ]:
import matplotlib.pyplot as plt
listX = [2016, 2022]                                  # 課本 ex10/plt01.py
listY = [0, 100000]
plt.plot(listX, listY, color='blue', ls='-.', lw=4, label='Sales volume')
plt.legend()
plt.show()

## 11-2　線條圖的細節：樣式、標題、座標、範圍（課本 10.2）
| 參數 | 意義 | 例子 |
|---|---|---|
| `color` | 顏色 | `'red'`、`'#1f77b4'` |
| `ls` / `linestyle` | 線條樣式 | `'-'`、`'--'`、`'-.'`、`':'` |
| `lw` | 線寬 | `2` |
| `marker` / `ms` | 端點符號／大小 | `'o'`、`'*'`、`'s'` |
| `label` | 圖例文字 | 搭配 `plt.legend()` |

In [ ]:
import matplotlib.pyplot as plt
years = [2017, 2018, 2019, 2020, 2021, 2022]                 # 課本 ex10/plt04.py
iphone = [43000, 31000, 70500, 68000, 85000, 24000]
asus   = [23000, 36000, 40500, 58000, 65000, 44000]
google = [13000, 26000, 50500, 68000, 75000, 54000]
plt.plot(years, iphone, color='blue', ls='-', lw=2, marker='o', ms=8, label='iPhone')
plt.plot(years, asus, color='red', ls='-.', lw=2, marker='*', ms=10, label='ASUS')
plt.plot(years, google, color='green', ls='--', lw=2, marker='s', ms=8, label='Google')
plt.title('手機歷年銷售量')
plt.xlim(2016, 2023); plt.ylim(0, 110000)
plt.xlabel('年度'); plt.ylabel('銷售量')
plt.legend(); plt.grid(True)
plt.show()

## 11-3　柱狀圖與圓餅圖（課本 10.3–10.4）

In [ ]:
import matplotlib.pyplot as plt
years = [2017, 2018, 2019, 2020, 2021, 2022]
iphone = [43000, 31000, 70500, 68000, 85000, 24000]
asus   = [23000, 36000, 40500, 58000, 65000, 44000]
plt.bar(years, iphone, label='iPhone')                      # 課本 ex10/plt07.py：疊加柱狀圖
plt.bar(years, asus, bottom=iphone, label='ASUS')
plt.title('手機歷年銷售量'); plt.xlabel('年度'); plt.ylabel('銷售量')
plt.legend(); plt.grid(True)
plt.show()

percent = [15.5, 18, 34.5, 7, 25]                             # 課本 ex10/plt09.py：圓餅圖
books = ['七龍珠', '火影忍者', '航海王', '第一神拳', '多啦A夢']
plt.pie(percent, labels=books, explode=(0, 0, 0.2, 0, 0.1), autopct='%3.1f%%', shadow=True, startangle=90)
plt.axis('equal')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.show()

> 🧪 **檢查怎麼運作**：檢查工具會重新執行你的程式格並檢視「畫了幾條線／幾根柱子、標題是什麼」，所以請把畫圖的程式寫在任務格裡，並用 `plt.title()`、`plt.xlabel()` 設定文字。

### 🎯 任務 11-1　我的第一張線條圖

畫出 `x = [1, 2, 3, 4, 5]`、`y = [2, 4, 6, 8, 10]` 的線條圖，紅色、虛線 `'--'`、圓點 marker `'o'`、標籤 `'y = 2x'`，標題 `我的第一張圖`，顯示圖例。

In [ ]:
# 🎯 任務 11-1　我的第一張線條圖（請保留這一行）
import matplotlib.pyplot as plt
x = [1, 2, 3, 4, 5]
y = [2, 4, 6, 8, 10]
plt.plot(???)
plt.title(???)
plt.legend()
plt.show()

In [ ]:
檢查("11-1")   # ◀ 執行這一格，看看任務 11-1 有沒有過關

### 🎯 任務 11-2　招生分數趨勢

碁峰科大 2018～2023 三群錄取分數：商管 `[500, 512, 430, 480, 490, 530]`、設計 `[430, 500, 510, 300, 320, 520]`、資電 `[330, 400, 410, 500, 520, 580]`。畫成 **3 條線**（各自 label），標題 `歷年錄取分數`，x 軸 `年度`、y 軸 `分數`，顯示圖例與格線。

In [ ]:
# 🎯 任務 11-2　招生分數趨勢（請保留這一行）
import matplotlib.pyplot as plt
years = [2018, 2019, 2020, 2021, 2022, 2023]
biz = [500, 512, 430, 480, 490, 530]
design = [430, 500, 510, 300, 320, 520]
it = [330, 400, 410, 500, 520, 580]
plt.plot(years, biz, marker='o', label='商管')
# 再畫設計、資電兩條線
plt.title(???)
plt.xlabel(???); plt.ylabel(???)
plt.legend(); plt.grid(True)
plt.show()

In [ ]:
檢查("11-2")   # ◀ 執行這一格，看看任務 11-2 有沒有過關

### 🎯 任務 11-3　成績柱狀圖

把 5 位學生 `names = ['小明', '小美', '阿華', '小芳', '大雄']` 的分數 `[78, 92, 55, 88, 61]` 畫成柱狀圖（`plt.bar`），標題 `期中考成績`，y 軸範圍 0～100。

In [ ]:
# 🎯 任務 11-3　成績柱狀圖（請保留這一行）
import matplotlib.pyplot as plt
names = ['小明', '小美', '阿華', '小芳', '大雄']
scores = [78, 92, 55, 88, 61]
plt.bar(???)
plt.title(???)
plt.ylim(???)
plt.show()

In [ ]:
檢查("11-3")   # ◀ 執行這一格，看看任務 11-3 有沒有過關

### 🎯 任務 11-4　招生員額圓餅圖

三群招生比例 `[25.2, 31.8, 43]`、標籤 `['商管', '設計', '資電']`。畫圓餅圖，顯示百分比（`autopct='%.1f%%'`），`資電` 那一塊凸出 0.1，標題 `招生員額比例`。

In [ ]:
# 🎯 任務 11-4　招生員額圓餅圖（請保留這一行）
import matplotlib.pyplot as plt
ratio = [25.2, 31.8, 43]
groups = ['商管', '設計', '資電']
plt.pie(???)
plt.title(???)
plt.axis('equal')
plt.show()

In [ ]:
檢查("11-4")   # ◀ 執行這一格，看看任務 11-4 有沒有過關

### 🎯 任務 11-5　儲存圖檔

畫出 `x = range(1, 11)` 與 `y = [i ** 2 for i in x]` 的線條圖，標題 `平方數`，並用 `plt.savefig('square.png')` 存成圖檔（要寫在 `plt.show()` **之前**）。

In [ ]:
# 🎯 任務 11-5　儲存圖檔（請保留這一行）
import matplotlib.pyplot as plt
x = list(range(1, 11))
y = [i ** 2 for i in x]
plt.plot(x, y, marker='o')
plt.title(???)
plt.savefig(???)
plt.show()

In [ ]:
檢查("11-5")   # ◀ 執行這一格，看看任務 11-5 有沒有過關

## 💡 挑戰題（不計分）
用 `plt.subplot(1, 2, 1)` 與 `plt.subplot(1, 2, 2)` 把柱狀圖和圓餅圖並排畫在同一張圖裡。

---
## 🔑 通關密語

全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：⚔️ B5 Boss 戰：銷售報表產生器** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/B5_boss_sales_report.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/